<a href="https://colab.research.google.com/github/gilbertoag2007/fiap-tech-challenge-fase3/blob/main/tech_challenge_fase_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AGRUPANDO AS 5 PARTES DO DATASET

In [8]:
# Clone do repositório no GITHUB
!git clone https://github.com/gilbertoag2007/fiap-tech-challenge-fase3.git

fatal: destination path 'fiap-tech-challenge-fase3' already exists and is not an empty directory.


In [9]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import pandas as pd
import re
import spacy
from pathlib import Path

!pip install -q pandas pyarrow openpyxl
!pip install pandas openpyxl spacy
!python -m spacy download pt_core_news_lg



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB ? eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# CARREGAMENTO DOS DADOS

In [10]:
"""
===========================================================================
OBJETIVO
===========================================================================

Este script reúne vários arquivos XLSX que foram divididos devido ao
limite de tamanho do GitHub (ex.: arquivos maiores que 80 MB).

Ao final será gerado um único dataset que servirá de entrada para as
etapas de:

1. Identificação de dados pessoais (PHI)
2. Anonimização
3. Limpeza dos dados
4. Fine-Tuning de uma LLM

Bibliotecas necessárias:

pip install pandas openpyxl

===========================================================================
"""

# ==========================================================================
# CONFIGURAÇÕES
# ==========================================================================

# Pasta onde estão os arquivos divididos
PASTA_DADOS = "/content/fiap-tech-challenge-fase3/data"

# Nome do arquivo de saída
ARQUIVO_FINAL = "dataset_medico_completo.xlsx"

# ==========================================================================
# Localiza todos os arquivos XLSX da pasta
# ==========================================================================

print("=" * 70)
print("LOCALIZANDO ARQUIVOS...")
print("=" * 70)

# Procura todos os arquivos .xlsx
arquivos = sorted(Path(PASTA_DADOS).glob("*.xlsx"))

# Verifica se encontrou arquivos
if len(arquivos) == 0:
    raise Exception("Nenhum arquivo XLSX encontrado.")

print(f"Foram encontrados {len(arquivos)} arquivos.\n")

# ==========================================================================
# Leitura dos arquivos
# ==========================================================================

print("=" * 70)
print("LENDO ARQUIVOS...")
print("=" * 70)

lista_dataframes = []

for arquivo in arquivos:

    print(f"Lendo {arquivo.name}")

    # Carrega o arquivo para um DataFrame
    df = pd.read_excel(
        arquivo,
        engine="openpyxl"
    )

    # Guarda o DataFrame em memória
    lista_dataframes.append(df)

# ==========================================================================
# Junta todos os DataFrames
# ==========================================================================

print("\nUnindo arquivos...")

dataframe_original = pd.concat(
    lista_dataframes,
    ignore_index=True
)

print("Arquivos unidos com sucesso!")



LOCALIZANDO ARQUIVOS...
Foram encontrados 4 arquivos.

LENDO ARQUIVOS...
Lendo dataset_medico_part001.xlsx
Lendo dataset_medico_part002.xlsx
Lendo dataset_medico_part003.xlsx
Lendo dataset_medico_part004.xlsx

Unindo arquivos...
Arquivos unidos com sucesso!


In [11]:
# ==========================================================================
# Limpeza básica
# ==========================================================================

print("\nRealizando limpeza inicial...")

# Remove linhas totalmente vazias
dataframe_original.dropna(
    how="all",
    inplace=True
)

# Remove registros duplicados
dataframe_original.drop_duplicates(
    inplace=True
)

# Reinicia a numeração do índice
dataframe_original.reset_index(
    drop=True,
    inplace=True
)



Realizando limpeza inicial...


In [12]:
# ==========================================================================
# REDUZ O TAMANHO DO DATASET PARA AGILIZAR OS TESTES EM DESENVOLVIMENTO
# ==========================================================================

dataframe_reduzido = dataframe_original.sample(frac=0.01, random_state=42)


In [13]:
# ==========================================================================
# Informações do dataset
# ==========================================================================

print("\nResumo do dataset")

print("-" * 60)

print(f"Quantidade de registros : {len(dataframe_reduzido):,}")

print(f"Quantidade de colunas   : {len(dataframe_reduzido.columns)}")

print("\nColunas encontradas:\n")

for coluna in dataframe_reduzido.columns:
    print(f" - {coluna}")

# ==========================================================================
# Verificação de valores nulos
# ==========================================================================

print("\nValores ausentes por coluna\n")

print(dataframe_reduzido.isnull().sum())

# ==========================================================================
# Estatísticas básicas
# ==========================================================================

print("\nPrimeiros registros:\n")

print(dataframe_reduzido.head())


Resumo do dataset
------------------------------------------------------------
Quantidade de registros : 3,841
Quantidade de colunas   : 6

Colunas encontradas:

 - id
 - pergunta_com_dados_pessoais
 - resposta_formatada
 - condicao
 - especialidade_medica
 - tipo_pergunta

Valores ausentes por coluna

id                             0
pergunta_com_dados_pessoais    0
resposta_formatada             0
condicao                       0
especialidade_medica           0
tipo_pergunta                  0
dtype: int64

Primeiros registros:

            id                        pergunta_com_dados_pessoais  \
56799   331651             Demência. O que pode levar à demência?   
299108  429542  Meu namorado disse que há um tempo atrás foi d...   
172048  338711  Estou em um relacionamento relativamente recen...   
21162    27410  O que pode causar zumbido pulsátil , sendo que...   
82076   117978  exercício físico diminui a glicemia e a glicos...   

                                       respost

In [14]:

# ==========================================================================
# Salva o dataset consolidado
# ==========================================================================

print("\nSalvando arquivo consolidado...")

dataframe_reduzido.to_excel(
    ARQUIVO_FINAL,
    index=False,
    engine="openpyxl"
)

print("\nArquivo salvo com sucesso!")

print(f"\nArquivo gerado: {ARQUIVO_FINAL}")



Salvando arquivo consolidado...

Arquivo salvo com sucesso!

Arquivo gerado: dataset_medico_completo.xlsx


# DECTECTAR PHI - Protected Health Information

In [15]:
# ============================================================
# IDENTIFICAÇÃO DE DADOS PESSOAIS EM DATASET MÉDICO
# ============================================================
#
# Objetivo:
#   Percorrer todas as colunas de um CSV/XLSX e identificar
#   possíveis informações pessoais ou sensíveis.
#
# Detecta:
#   - CPF
#   - E-mail
#   - Telefone
#   - CEP
#   - Datas
#   - Nomes de pessoas
#   - Organizações
#   - Localidades
#
# IMPORTANTE:
#   Este código NÃO substitui uma ferramenta de anonimização.
#   Ele é uma primeira etapa de análise/curadoria do dataset.
# ============================================================


# ============================================================
# 1. CARREGAR MODELO NLP EM PORTUGUÊS
# ============================================================

# O modelo reconhece entidades como:
#
# PERSON       -> pessoa
# ORG          -> organização
# LOC/GPE      -> localidade
#
# Instalação:
#
# !python -m spacy download pt_core_news_lg

nlp = spacy.load("pt_core_news_lg")


# ============================================================
# 2. EXPRESSÕES REGULARES
# ============================================================

PADROES = {

    # CPF
    "CPF": re.compile(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b"
    ),

    # E-mail
    "EMAIL": re.compile(
        r"\b[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),

    # Telefone brasileiro
    "TELEFONE": re.compile(
        r"(?<!\d)"
        r"(?:\+55\s?)?"
        r"(?:\(?\d{2}\)?\s?)?"
        r"(?:9?\d{4})[-\s]?\d{4}"
        r"(?!\d)"
    ),

    # CEP
    "CEP": re.compile(
        r"\b\d{5}-?\d{3}\b"
    ),

    # Datas no formato dd/mm/yyyy ou dd-mm-yyyy
    "DATA": re.compile(
        r"\b(?:0?[1-9]|[12]\d|3[01])"
        r"[/\-]"
        r"(?:0?[1-9]|1[0-2])"
        r"[/\-]"
        r"(?:19|20)\d{2}\b"
    ),

    # RG genérico
    #
    # Este padrão é propositalmente mais conservador.
    "RG": re.compile(
        r"\b\d{1,2}\.?\d{3}\.?\d{3}-?[0-9Xx]\b"
    )
}


# ============================================================
# 3. FUNÇÃO PARA DETECTAR REGEX
# ============================================================

def detectar_regex(texto):

    encontrados = []

    for tipo, padrao in PADROES.items():

        ocorrencias = padrao.findall(texto)

        if ocorrencias:

            encontrados.append(tipo)

    return encontrados


# ============================================================
# 4. FUNÇÃO PARA DETECTAR ENTIDADES COM spaCy
# ============================================================

def detectar_entidades(texto):

    entidades = []

    doc = nlp(texto)

    for entidade in doc.ents:

        # Pessoa
        if entidade.label_ == "PER":

            entidades.append("NOME_PESSOA")

        # Organização
        elif entidade.label_ == "ORG":

            entidades.append("ORGANIZACAO")

        # Localização
        elif entidade.label_ in ["LOC", "GPE"]:

            entidades.append("LOCALIZACAO")

    return list(set(entidades))


# ============================================================
# 5. ANALISAR UMA CÉLULA
# ============================================================

def analisar_texto(valor):

    if pd.isna(valor):

        return []

    texto = str(valor)

    tipos = []

    # -------------------------------
    # Regex
    # -------------------------------

    tipos.extend(
        detectar_regex(texto)
    )

    # -------------------------------
    # NER
    # -------------------------------

    tipos.extend(
        detectar_entidades(texto)
    )

    # Remover duplicidades
    tipos = sorted(
        list(set(tipos))
    )

    return tipos




Analisando coluna: pergunta_com_dados_pessoais
Analisando coluna: resposta_formatada


RELATÓRIO DE DADOS PESSOAIS
                     coluna  total_registros  registros_com_dados_pessoais  percentual_suspeito                                                    tipos_detectados                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [17]:
# ============================================================
# ANÁLISE DE DADOS PESSOAIS / PII / PHI
# ============================================================
#
# O código:
#
# 1. Percorre as colunas definidas em "colunas_analisar"
# 2. Analisa cada registro
# 3. Identifica os tipos de dados pessoais
# 4. Conta quantos registros possuem cada tipo
# 5. Calcula o percentual de cada tipo
# 6. Mostra o resultado diretamente no notebook
#
# IMPORTANTE:
# Um mesmo registro pode possuir mais de um tipo de dado.
#
# Exemplo:
#
# "João da Silva, CPF 123.456.789-00,
#  telefone (21)99999-9999"
#
# Será contabilizado como:
#
# NOME_PESSOA -> 1
# CPF         -> 1
# TELEFONE    -> 1
#
# Porém "registros_com_dados_pessoais" será incrementado
# somente uma vez.
# ============================================================


# ============================================================
# 1. DEFINIR AS COLUNAS QUE SERÃO ANALISADAS
# ============================================================

# Informe aqui as duas (ou mais) colunas que deseja analisar.

colunas_analisar = [
    "pergunta_com_dados_pessoais",
    "resposta_formatada"
]


# ============================================================
# 2. LISTA PARA ARMAZENAR OS RESULTADOS
# ============================================================

resultados = []


# ============================================================
# 3. PERCORRER AS COLUNAS
# ============================================================

for coluna in colunas_analisar:

    print("\n")
    print("=" * 90)
    print(f"ANALISANDO COLUNA: {coluna}")
    print("=" * 90)


    # --------------------------------------------------------
    # Quantidade total de registros da coluna
    # --------------------------------------------------------

    total_registros = len(dataframe_reduzido)


    # --------------------------------------------------------
    # Quantidade de registros que possuem pelo menos
    # um tipo de dado pessoal
    # --------------------------------------------------------

    registros_com_dados = 0


    # --------------------------------------------------------
    # Dicionário para armazenar a quantidade encontrada
    # de cada tipo de dado pessoal
    #
    # Exemplo:
    #
    # {
    #     "CPF": 10,
    #     "NOME_PESSOA": 150,
    #     "TELEFONE": 30
    # }
    # --------------------------------------------------------

    contadores = {}


    # --------------------------------------------------------
    # Armazena alguns exemplos encontrados.
    #
    # Limitamos a 5 para não imprimir uma quantidade
    # enorme de dados potencialmente sensíveis.
    # --------------------------------------------------------

    exemplos = []


    # ========================================================
    # 4. PERCORRER TODOS OS REGISTROS DA COLUNA
    # ========================================================

    for indice, valor in dataframe_reduzido[coluna].items():

        # ----------------------------------------------------
        # Executa sua função de análise
        #
        # Exemplo de retorno:
        #
        # ["CPF", "NOME_PESSOA"]
        # ----------------------------------------------------

        tipos = analisar_texto(valor)


        # ----------------------------------------------------
        # Verifica se foi encontrado algum dado pessoal
        # ----------------------------------------------------

        if tipos:

            # Conta o registro apenas uma vez
            registros_com_dados += 1


            # =================================================
            # 5. CONTAR CADA TIPO DE DADO INDIVIDUALMENTE
            # =================================================

            for tipo in tipos:

                # Se for a primeira ocorrência daquele tipo,
                # inicializa o contador.
                if tipo not in contadores:

                    contadores[tipo] = 0


                # Incrementa o contador
                contadores[tipo] += 1


            # =================================================
            # 6. GUARDAR ATÉ 5 EXEMPLOS
            # =================================================

            if len(exemplos) < 5:

                exemplos.append({

                    "linha": indice + 2,

                    "valor": str(valor)[:200],

                    "tipos": ", ".join(tipos)

                })


    # ========================================================
    # 7. CALCULAR PERCENTUAL DE REGISTROS SUSPEITOS
    # ========================================================

    if total_registros > 0:

        percentual_suspeito = (
            registros_com_dados /
            total_registros
        ) * 100

    else:

        percentual_suspeito = 0


    # ========================================================
    # 8. EXIBIR RESUMO DA COLUNA
    # ========================================================

    print(f"\nTotal de registros: "
          f"{total_registros:,}")

    print(f"Registros com dados pessoais: "
          f"{registros_com_dados:,}")

    print(f"Percentual de registros suspeitos: "
          f"{percentual_suspeito:.2f}%")


    # ========================================================
    # 9. EXIBIR RESULTADO POR TIPO
    # ========================================================

    print("\n")
    print("DADOS PESSOAIS IDENTIFICADOS")
    print("-" * 90)

    # Verifica se algum tipo foi encontrado

    if contadores:

        # Cabeçalho da tabela
        print(
            f"{'TIPO':<25}"
            f"{'QUANTIDADE':>15}"
            f"{'PERCENTUAL':>18}"
        )

        print("-" * 90)


        # Percorre os tipos encontrados
        for tipo, quantidade in sorted(
            contadores.items()
        ):

            # ----------------------------------------------
            # Percentual do tipo em relação ao total
            # de registros da coluna
            # ----------------------------------------------

            if total_registros > 0:

                percentual_tipo = (
                    quantidade /
                    total_registros
                ) * 100

            else:

                percentual_tipo = 0


            # ----------------------------------------------
            # Exibir resultado
            # ----------------------------------------------

            print(
                f"{tipo:<25}"
                f"{quantidade:>15,}"
                f"{percentual_tipo:>17.2f}%"
            )


    else:

        print(
            "Nenhum dado pessoal foi identificado."
        )


    # ========================================================
    # 10. MOSTRAR EXEMPLOS
    # ========================================================
    #
    # ATENÇÃO:
    # Em datasets médicos reais, recomendo NÃO imprimir
    # o conteúdo completo da célula.
    #
    # Para testes com dados sintéticos, os exemplos podem
    # ser úteis para validar o detector.
    # ========================================================

    if exemplos:

        print("\n")
        print("EXEMPLOS ENCONTRADOS")
        print("-" * 90)

        for exemplo in exemplos:

            print(
                f"Linha: {exemplo['linha']}"
            )

            print(
                f"Tipos: {exemplo['tipos']}"
            )

            print(
                f"Valor: {exemplo['valor']}"
            )

            print("-" * 90)


    # ========================================================
    # 11. ARMAZENAR RESULTADO DA COLUNA
    # ========================================================

    resultados.append({

        "coluna": coluna,

        "total_registros": total_registros,

        "registros_com_dados_pessoais":
            registros_com_dados,

        "percentual_suspeito":
            round(percentual_suspeito, 2),

        "quantidades_por_tipo":
            contadores,

        "exemplos":
            exemplos

    })


# ============================================================
# 12. RESUMO FINAL DE TODAS AS COLUNAS
# ============================================================

print("\n\n")
print("=" * 90)
print("RESUMO FINAL DA ANÁLISE")
print("=" * 90)


for resultado in resultados:

    print("\n")
    print(f"COLUNA: {resultado['coluna']}")

    print(
        f"Total de registros: "
        f"{resultado['total_registros']:,}"
    )

    print(
        f"Registros com dados pessoais: "
        f"{resultado['registros_com_dados_pessoais']:,}"
    )

    print(
        f"Percentual suspeito: "
        f"{resultado['percentual_suspeito']:.2f}%"
    )

    print("\nTipos identificados:")

    for tipo, quantidade in sorted(
        resultado["quantidades_por_tipo"].items()
    ):

        percentual = (
            quantidade /
            resultado["total_registros"]
        ) * 100

        print(
            f"  {tipo:<25}"
            f"{quantidade:>10,} "
            f"({percentual:.2f}%)"
        )



ANALISANDO COLUNA: pergunta_com_dados_pessoais

Total de registros: 3,841
Registros com dados pessoais: 1,329
Percentual de registros suspeitos: 34.60%


DADOS PESSOAIS IDENTIFICADOS
------------------------------------------------------------------------------------------
TIPO                          QUANTIDADE        PERCENTUAL
------------------------------------------------------------------------------------------
CEP                                    4             0.10%
CPF                                  172             4.48%
DATA                                 172             4.48%
LOCALIZACAO                          642            16.71%
NOME_PESSOA                          651            16.95%
ORGANIZACAO                          254             6.61%
RG                                     5             0.13%
TELEFONE                              14             0.36%


EXEMPLOS ENCONTRADOS
-------------------------------------------------------------------------------